# DB_MALARIA ablation: metric vs deltaprop hard-negative fraction

Pulls the single run from the `ablation_db_malaria_frac_hard` wandb project, recovers the per-(frac_hard, model) rows from its history, and plots deltaprop across hard-negative fractions with the chemprop baseline as a horizontal reference line.

Each `wandb.log(row)` call in the ablation became a separate history step, so we read the full history with `scan_history()` and split on the `model` column ourselves. chemprop is invariant to `frac_hard`, so it appears once with `frac_hard = NaN`.

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = "ablations"
RUN_PREFIX = "db_malaria_frac_hard"
RUN_NAME = None  # set to an exact run name to pin a specific run; otherwise use the latest
METRIC = "test_average_precision"  # swap for test_mcc, test_f1, test_roc_auc, ...

api = wandb.Api()
runs = api.runs(f"{api.default_entity}/{PROJECT}")
if RUN_NAME is not None:
    run = next(r for r in runs if r.name == RUN_NAME)
else:
    # Runs are named '<prefix>_<timestamp>'; pick the most recent matching run.
    matches = [r for r in runs if r.name.startswith(RUN_PREFIX)]
    run = max(matches, key=lambda r: r.created_at)
run.name, run.id

In [ ]:
# Full, unsampled history -> one row per (frac_hard, model).
df = pd.DataFrame(run.scan_history())
df = df[["frac_hard", "model", "n_train", METRIC]].sort_values(["model", "frac_hard"])
df

In [ ]:
delta = df[df["model"] == "deltaprop"].sort_values("frac_hard")
chemprop = df[df["model"] == "chemprop"]

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(delta["frac_hard"], delta[METRIC], marker="o", label="deltaprop")

# if len(chemprop):
#     ax.axhline(
#         chemprop[METRIC].iloc[0],
#         color="gray",
#         linestyle="--",
#         label="chemprop (baseline)",
#     )

ax.set_xticks(delta["frac_hard"])
ax.set_xlabel("deltaprop hard-negative fraction")
ax.set_ylabel(METRIC)
ax.set_title("DB_MALARIA hard-negative fraction ablation (SCAFFOLD split)")
ax.grid(True)
ax.legend()
fig.tight_layout()
# fig.savefig("db_malaria_hard_negatives_ablation.png", dpi=150)
plt.show()